# 07. Reimplement a Method from the Paper

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook teaches the optimized process for building a clean implementation when released code should not be used directly.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Create a complete method specification
- Translate architecture diagrams into modules
- Unit-test components
- Treat missing details as experiments
- Compare implementation behavior against reported evidence


## Mind map

```mermaid
mindmap
  root((Reimplementation))
    Specification
      Data
      Architecture
      Loss
      Training
    Implement
      Small modules
      Shapes
      Tests
    Validate
      Tiny overfit
      Known loss cases
      Curves
    Unknowns
      Document
      Experiment
      Sensitivity
    Compare
      Paper metrics
      Trends
      Examples

```


## 1. When reimplementation is the better choice

Reimplement when:
- no code;
- license prevents reuse;
- repository is incomplete;
- dependencies are irrecoverably fragile;
- paper/code disagree substantially;
- your task differs so much that adapting old infrastructure is harder than writing a clean baseline.

Reimplementation means **faithfully implementing the method you need**, not guessing missing details silently.


## 2. Build the specification before the model

Create four tables:

### Data
input shape, target, normalization, split, augmentation

### Architecture
stage, operation, channels, resolution

### Objective
every loss term, equation, coefficient, reduction

### Training
optimizer, LR, scheduler, batch, epochs, checkpoint

Unknown items are marked **unknown** and become sensitivity experiments.


## 3. Translate an architecture figure into modules

Example specification:

```text
Input: 1×256×256
Block1: Conv(1→32,3,pad1) + ReLU ×2
Pool:   MaxPool2
Block2: Conv(32→64,3,pad1) + ReLU ×2
...
Output: Conv(32→1,1)
```

Only after this table is consistent should you implement classes.


In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

block = DoubleConv(1, 32)
x = torch.randn(2, 1, 256, 256)
print(block(x).shape)


## 4. Unit-test components before training

Test:
- output shapes;
- finite values (no NaN/Inf);
- loss on known cases;
- transform alignment between image and mask;
- deterministic validation transform;
- model can overfit tiny data.

This turns debugging from "training does not work" into a small, local question.


## 5. Unknown paper details become controlled experiments

If the paper does not state whether normalization is global or per-image, do not hide your choice.

Document:

```text
Unknown: normalization
Implementation A: per-image percentile
Implementation B: global mean/std
Selection rule: validation Dice on fixed specimen split
```

This is scientifically stronger than pretending the paper was complete.


## 6. Compare your implementation against the paper at multiple levels

- parameter count/order of magnitude;
- tensor shapes;
- training loss behavior;
- validation metric range;
- qualitative examples;
- inference time if relevant;
- ablation trend.

Exact matching may be impossible if data/training randomness are unavailable, but major discrepancies require investigation.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
